<a href="https://colab.research.google.com/github/lcandau/histopathology-clip-lab/blob/exp_06_biomed_text/experiments/exp_07_ood_eval/CLIP_Biomed_OOD_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# exp_07 — Biomed CLIP variants OOD evaluation on NCT-CRC-HE-7K

Loops over the 6 `(text_encoder, prompt_strategy)` checkpoints produced by
`exp_06_biomed_text/CLIP_Biomed_text.ipynb` and evaluates each on NCT-CRC.

The model architecture here is `CLIPModel_HF` (HuggingFace text encoder + KerasHub ResNet50 image encoder + projection heads), not the standard `CLIPModel` used by the baseline/stain OOD eval. So this is a sibling of `CLIP_OOD_eval.ipynb` rather than a VARIANT setting.

**Ablation grid:**
| | name_only | composed |
|---|---|---|
| bert-base-uncased | `bert_name_only` | `bert_composed` |
| dmis-lab/biobert-v1.1 | `biobert_name_only` | `biobert_composed` |
| microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract | `pubmed_name_only` | `pubmed_composed` |

For each cell we (1) load the trained `CLIPModel_HF` weights, (2) re-encode the 2 colon-class prompts using the matching tokenizer + strategy, (3) classify NCT NORM/TUM by argmax over `colon_indices`, (4) save per-variant metrics + UMAPs. Cells with missing checkpoints are skipped with a warning.


## 0 — Bootstrap


In [ ]:
# --- Cell 0: bootstrap ---
import os, sys, subprocess
os.environ["KERAS_BACKEND"] = "tensorflow"

REPO_URL = "https://github.com/lcandau/histopathology-clip-lab.git"
REPO_DIR = "/content/histopathology-clip-lab"
BRANCH   = "exp_06_biomed_text"
IN_COLAB = "google.colab" in sys.modules


def _clone_with_fallback(branch):
    try:
        subprocess.run(
            ["git", "clone", "-b", branch, REPO_URL, REPO_DIR],
            check=True, capture_output=True,
        )
        print(f"Cloned branch {branch!r}")
        return branch
    except subprocess.CalledProcessError:
        print(f"Branch {branch!r} not found; cloning main")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
        return "main"


if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        active = _clone_with_fallback(BRANCH)
    else:
        fetch = subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], capture_output=True)
        if fetch.returncode == 0:
            subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
            subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
            active = BRANCH
        else:
            subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
            subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
            subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
            active = "main"
    print(f"Active branch: {active}")
    subprocess.run([
        "pip", "install", "-q",
        "tensorflow==2.18.0", "keras==3.7.0", "keras-hub==0.18.1",
        "transformers==4.46.0", "umap-learn", "kagglehub",
        "scikit-learn", "matplotlib", "pandas", "Pillow",
    ], check=True)
    subprocess.run(["pip", "uninstall", "-y", "-q", "jax", "jaxlib"], check=False)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
else:
    LOCAL_REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    if LOCAL_REPO_DIR not in sys.path:
        sys.path.insert(0, LOCAL_REPO_DIR)

print("In Colab:", IN_COLAB)
print("sys.path[0]:", sys.path[0])
print("KERAS_BACKEND:", os.environ.get("KERAS_BACKEND"))


## 1 — Imports


In [ ]:
# --- Cell 1: imports ---
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import json
import math
import shutil
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
import keras
import keras_hub

from transformers import AutoTokenizer, TFAutoModel

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score,
)

from src.utils.paths import is_colab, repo_root, drive_root, run_dir, results_dir
from src.utils.repro import set_global_seed, enable_op_determinism
from src.data.lc25000 import CLASS_INFO, NUM_CLASSES, CLASS_NAMES, INDEX_TO_NAME, ID_TO_INDEX
from src.prompts.templates import build_prompts, PROMPT_STRATEGIES


## 2 — Configuration + ablation grid


In [ ]:
# --- Cell 2: config ---
EXPERIMENT = "exp_07_ood_eval"
SEED = 42
set_global_seed(SEED)
enable_op_determinism()

IMG_SIZE = 224
MAX_LEN = 32      # matches exp_06_biomed_text training notebook
EMBED_DIM = 256
INIT_TEMP = 0.07
BATCH_SIZE = 64

# 6-cell ablation grid (must match exp_06_biomed_text/CLIP_Biomed_text.ipynb).
ENCODERS = {
    "bert":    "bert-base-uncased",
    "biobert": "dmis-lab/biobert-v1.1",
    "pubmed":  "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
}
STRATEGIES = ("name_only", "composed")
ABLATION_GRID = [
    (f"{enc_key}_{strategy}", enc_id, strategy)
    for enc_key, enc_id in ENCODERS.items()
    for strategy in STRATEGIES
]

# Checkpoints live under exp_06_biomed_text/<cell_name>/weights.weights.h5 on Drive.
def _ckpt(cell_name):
    return run_dir("exp_06_biomed_text", cell_name) / "weights.weights.h5"

# NCT-CRC-HE-7K.
NCT_URL  = "https://zenodo.org/records/1214456/files/CRC-VAL-HE-7K.zip"
NCT_NAME = "CRC-VAL-HE-7K"
NCT_TO_LC25000 = {
    "NORM": "benign colon tissue",
    "TUM":  "colon adenocarcinoma",
}
DRIVE_OOD_ZIP   = drive_root() / "ood_datasets" / f"{NCT_NAME}.zip"
LOCAL_OOD_ROOT  = Path("/content/ood_data") if is_colab() else Path("/tmp/ood_data")
LOCAL_OOD_DIR   = LOCAL_OOD_ROOT / NCT_NAME

METRICS_DIR = results_dir("metrics") / EXPERIMENT
CM_DIR      = results_dir("confusion_matrices") / EXPERIMENT
PLOTS_DIR   = results_dir("plots") / EXPERIMENT
for d in (METRICS_DIR, CM_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Ablation grid:")
for cell_name, enc_id, strategy in ABLATION_GRID:
    ckpt = _ckpt(cell_name)
    exists = ckpt.is_file()
    print(f"  {cell_name:22s}  enc={enc_id:60s}  strat={strategy:10s}  ckpt={'OK' if exists else 'MISSING'}")


## 3 — Acquire NCT-CRC-HE-7K


In [ ]:
# --- Cell 3: download / extract NCT-CRC ---
def _ensure_nct_zip(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists(): return path
    DRIVE_OOD_ZIP.parent.mkdir(parents=True, exist_ok=True)
    if DRIVE_OOD_ZIP.exists():
        shutil.copy(DRIVE_OOD_ZIP, path)
        return path
    print(f"Downloading NCT-CRC from {NCT_URL} ...")
    urllib.request.urlretrieve(NCT_URL, path)
    shutil.copy(path, DRIVE_OOD_ZIP)
    return path


nct_zip_local = LOCAL_OOD_ROOT / f"{NCT_NAME}.zip"
_ensure_nct_zip(nct_zip_local)

if not LOCAL_OOD_DIR.exists() or not any(LOCAL_OOD_DIR.iterdir()):
    print(f"Extracting {nct_zip_local} -> {LOCAL_OOD_ROOT}")
    with zipfile.ZipFile(nct_zip_local) as zf:
        zf.extractall(LOCAL_OOD_ROOT)

class_dirs = sorted(p for p in LOCAL_OOD_DIR.iterdir() if p.is_dir())
print("\nClasses on disk:")
for cd in class_dirs:
    print(f"  {cd.name:>5s}  ({len(list(cd.iterdir()))} tiles)")


## 4 — Discover OOD records (NORM + TUM only)


In [ ]:
# --- Cell 4: discover OOD records ---
ood_paths = []
ood_nct_labels = []
for nct_cls in NCT_TO_LC25000:
    class_dir = LOCAL_OOD_DIR / nct_cls
    for f in sorted(class_dir.iterdir()):
        if f.suffix.lower() in {".tif", ".tiff", ".png", ".jpg", ".jpeg"}:
            ood_paths.append(str(f))
            ood_nct_labels.append(nct_cls)

ood_paths = np.asarray(ood_paths)
ood_nct_labels = np.asarray(ood_nct_labels)
ood_lc_indices = np.array([
    ID_TO_INDEX[next(c.id for c in CLASS_INFO if c.name == NCT_TO_LC25000[lab])]
    for lab in ood_nct_labels
], dtype=np.int32)

COLON_IDX = sorted({
    ID_TO_INDEX[next(c.id for c in CLASS_INFO if c.name == NCT_TO_LC25000[lab])]
    for lab in NCT_TO_LC25000
})

print(f"Total OOD tiles: {len(ood_paths)}  (NORM={int((ood_nct_labels=='NORM').sum())}, TUM={int((ood_nct_labels=='TUM').sum())})")
print(f"Colon class indices: {COLON_IDX}")


## 5 — Model + text-encoder helpers

Mirror of `exp_06_biomed_text/CLIP_Biomed_text.ipynb` Cell 11 — `CLIPModel_HF` consumes a HuggingFace text backbone alongside the frozen KerasHub ResNet50.


In [ ]:
# --- Cell 5: CLIPModel_HF + text encoder factory ---
def load_text_encoder(encoder_id):
    tokenizer = AutoTokenizer.from_pretrained(encoder_id)
    try:
        tf_model = TFAutoModel.from_pretrained(encoder_id, from_pt=False)
    except (OSError, EnvironmentError, TypeError, ValueError):
        tf_model = TFAutoModel.from_pretrained(encoder_id, from_pt=True)
    tf_model.trainable = False
    return tf_model, tokenizer, tf_model.config.hidden_size


def tokenize_prompts(tokenizer, prompts, max_len=MAX_LEN):
    out = tokenizer(prompts, padding="max_length", truncation=True,
                    max_length=max_len, return_tensors="tf")
    return {k: tf.cast(v, tf.int32) for k, v in out.items()}


class CLIPModel_HF(keras.Model):
    def __init__(self, img_backbone, text_backbone, hidden_size,
                 embed_dim=EMBED_DIM, init_temp=INIT_TEMP, **kwargs):
        super().__init__(**kwargs)
        self.img_backbone = img_backbone
        self.text_backbone = text_backbone
        self.hidden_size = hidden_size
        self.img_projection = keras.Sequential([
            keras.layers.Dense(embed_dim, use_bias=False, dtype="float32"),
            keras.layers.LayerNormalization(dtype="float32"),
        ], name="img_projection")
        self.text_projection = keras.Sequential([
            keras.layers.Dense(embed_dim, use_bias=False, dtype="float32"),
            keras.layers.LayerNormalization(dtype="float32"),
        ], name="text_projection")
        self.logit_scale = self.add_weight(
            name="logit_scale", shape=(),
            initializer=keras.initializers.Constant(math.log(1.0 / init_temp)),
            trainable=True, dtype="float32",
        )
        self.loss_tracker = keras.metrics.Mean(name="loss")

    @property
    def metrics(self):
        return [self.loss_tracker]

    def encode_image(self, images, training=False):
        features = self.img_backbone(images, training=False)
        if features.shape.rank == 4:
            features = tf.reduce_mean(features, axis=[1, 2])
        x = self.img_projection(features)
        return tf.math.l2_normalize(x, axis=-1)

    def encode_text(self, token_dict, training=False):
        kwargs = {"input_ids": token_dict["input_ids"],
                  "attention_mask": token_dict["attention_mask"]}
        if "token_type_ids" in token_dict:
            kwargs["token_type_ids"] = token_dict["token_type_ids"]
        out = self.text_backbone(**kwargs, training=False)
        pooled = getattr(out, "pooler_output", None)
        if pooled is None:
            pooled = out.last_hidden_state[:, 0, :]
        x = self.text_projection(pooled)
        return tf.math.l2_normalize(x, axis=-1)

    def call(self, inputs, training=False):
        images, token_dict = inputs
        return self.encode_image(images, training=training), self.encode_text(token_dict, training=training)


# Image backbone — frozen, shared across all 6 cells.
image_backbone = keras_hub.models.Backbone.from_preset("resnet_50_imagenet")
image_backbone.trainable = False
print("Image backbone loaded (frozen).")


## 6 — Encode OOD tiles ONCE per (encoder, strategy) cell

For each ablation cell we re-instantiate the text encoder, load the trained projection-head weights, and run inference. Image encoding is per-cell because the projection head is what changes — the ResNet50 backbone output is constant.

We also save the post-projection image embeddings for the UMAP later.


In [ ]:
# --- Cell 6: main eval loop ---
def encode_image_paths(model, paths, batch_size=BATCH_SIZE):
    out = []
    for start in range(0, len(paths), batch_size):
        chunk = paths[start:start + batch_size]
        arrs = []
        for p in chunk:
            img = Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
            arrs.append(np.asarray(img, dtype=np.float32) / 255.0)
        emb = model.encode_image(tf.constant(np.stack(arrs)), training=False).numpy()
        out.append(emb)
    return np.concatenate(out, axis=0)


all_results = {}
all_image_embeddings = {}
all_prompt_embeddings = {}

for cell_name, encoder_id, strategy in ABLATION_GRID:
    ckpt = _ckpt(cell_name)
    print(f"\n========== {cell_name} ==========")
    if not ckpt.is_file():
        print(f"  SKIP — checkpoint missing at {ckpt}")
        continue

    text_backbone, tokenizer, hidden = load_text_encoder(encoder_id)
    model = CLIPModel_HF(image_backbone, text_backbone, hidden)

    # Warm-up forward pass so weights materialise before load.
    dummy_imgs = tf.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32)
    dummy_tokens = tokenize_prompts(tokenizer, ["benign colon tissue"])
    _ = model((dummy_imgs, dummy_tokens), training=False)
    model.load_weights(str(ckpt))
    print(f"  Loaded weights from {ckpt}")

    # Encode the 5 class prompts using this cell's strategy (we'll restrict argmax to colon at scoring time).
    prompts = build_prompts(CLASS_NAMES, strategy)
    toks_all = tokenize_prompts(tokenizer, prompts)
    prompt_embeddings_all = model.encode_text(toks_all, training=False).numpy()

    # Encode all OOD tiles.
    ood_emb = encode_image_paths(model, ood_paths)
    sims_all = ood_emb @ prompt_embeddings_all.T
    sims_colon = sims_all[:, COLON_IDX]
    pred_within_colon = sims_colon.argmax(axis=1)
    y_pred = np.array([COLON_IDX[i] for i in pred_within_colon], dtype=np.int32)
    y_true = ood_lc_indices

    labels = COLON_IDX
    target_names = [INDEX_TO_NAME[i] for i in labels]
    acc = accuracy_score(y_true, y_pred)
    bal = balanced_accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    wf1 = f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    per_class = classification_report(y_true, y_pred, labels=labels,
                                       target_names=target_names,
                                       output_dict=True, zero_division=0)

    cm_raw = confusion_matrix(y_true, y_pred, labels=labels)
    row_sums = cm_raw.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_raw, row_sums, where=row_sums > 0,
                        out=np.zeros_like(cm_raw, dtype=np.float64))

    payload = {
        "experiment":        EXPERIMENT,
        "variant":           cell_name,
        "encoder_id":        encoder_id,
        "prompt_strategy":   strategy,
        "checkpoint":        str(ckpt),
        "ood_dataset":       NCT_NAME,
        "n_tiles":           int(len(ood_paths)),
        "accuracy":          float(acc),
        "balanced_accuracy": float(bal),
        "macro_f1":          float(mf1),
        "weighted_f1":       float(wf1),
        "per_class":         per_class,
        "colon_indices":     labels,
        "target_names":      target_names,
    }
    (METRICS_DIR / f"{cell_name}_ood_classification.json").write_text(json.dumps(payload, indent=2))
    np.save(CM_DIR / f"{cell_name}_ood_cm_raw.npy",  cm_raw)
    np.save(CM_DIR / f"{cell_name}_ood_cm_norm.npy", cm_norm)

    print(f"  acc={acc:.4f}  bal={bal:.4f}  macro_F1={mf1:.4f}")
    all_results[cell_name] = payload
    all_image_embeddings[cell_name] = ood_emb
    all_prompt_embeddings[cell_name] = prompt_embeddings_all

print(f"\nEvaluated {len(all_results)} / {len(ABLATION_GRID)} cells")


## 7 — Comparison table


In [ ]:
# --- Cell 7: comparison table ---
rows = []
for cell_name, encoder_id, strategy in ABLATION_GRID:
    if cell_name not in all_results:
        continue
    m = all_results[cell_name]
    rows.append({
        "encoder":           next(k for k, v in ENCODERS.items() if v == encoder_id),
        "strategy":          strategy,
        "accuracy":          m["accuracy"],
        "balanced_accuracy": m["balanced_accuracy"],
        "macro_f1":          m["macro_f1"],
        "weighted_f1":       m["weighted_f1"],
    })
summary_df = pd.DataFrame(rows)
print("OOD macro F1 across the (encoder, prompt) grid:")
print(summary_df.round(4).to_string(index=False))

pivot = summary_df.pivot(index="encoder", columns="strategy", values="macro_f1")
pivot = pivot.reindex(index=[k for k in ENCODERS], columns=list(STRATEGIES))
print("\nMacro F1 grid (rows=encoder, cols=strategy):")
print(pivot.round(4).to_string())

summary_df.to_csv(METRICS_DIR / "biomed_ood_summary.csv", index=False)
pivot.to_csv(METRICS_DIR / "biomed_ood_macro_f1_grid.csv")
print(f"\nSaved {METRICS_DIR / 'biomed_ood_summary.csv'}")


## 8 — 6-panel UMAP grid (one per ablation cell)


In [ ]:
# --- Cell 8: UMAP grid ---
import umap

n_rows = len(ENCODERS)
n_cols = len(STRATEGIES)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows))
axes = np.atleast_2d(axes)
if n_rows == 1: axes = axes.reshape(1, -1)
if n_cols == 1: axes = axes.reshape(-1, 1)

BENIGN_COLOR = "#08519c"
ADENO_COLOR  = "#a50f15"

for row_i, enc_key in enumerate(ENCODERS):
    for col_i, strategy in enumerate(STRATEGIES):
        cell_name = f"{enc_key}_{strategy}"
        ax = axes[row_i][col_i]
        if cell_name not in all_results:
            ax.text(0.5, 0.5, f"{cell_name}\nno checkpoint", ha="center", va="center",
                    transform=ax.transAxes, fontsize=11, color="grey")
            ax.set_xticks([]); ax.set_yticks([])
            continue
        ood_emb = all_image_embeddings[cell_name]
        prompt_emb_colon = all_prompt_embeddings[cell_name][COLON_IDX]
        combined = np.vstack([ood_emb, prompt_emb_colon])
        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=SEED)
        coords = reducer.fit_transform(combined)
        ood_2d, prompt_2d = coords[:len(ood_emb)], coords[len(ood_emb):]

        for nct_cls, color in (("NORM", BENIGN_COLOR), ("TUM", ADENO_COLOR)):
            mask = (ood_nct_labels == nct_cls)
            ax.scatter(ood_2d[mask, 0], ood_2d[mask, 1], s=6, alpha=0.5,
                       c=color, marker="^", label=f"NCT {nct_cls}")
        for k, idx in enumerate(COLON_IDX):
            color = BENIGN_COLOR if idx == 3 else ADENO_COLOR
            ax.scatter(prompt_2d[k, 0], prompt_2d[k, 1], s=320, marker="*",
                       c=color, edgecolors="white", linewidths=1.6, zorder=6)
        ax.set_xticks([]); ax.set_yticks([])
        mf1 = all_results[cell_name]["macro_f1"]
        ax.set_title(f"{cell_name}  (F1={mf1:.3f})", fontsize=11)
        if row_i == 0 and col_i == 0:
            ax.legend(fontsize=8, loc="best", framealpha=0.85)

fig.suptitle("Biomed CLIP variants — NCT-CRC-HE-7K OOD UMAPs",
             fontsize=14, y=1.005)
plt.tight_layout()
out_path = PLOTS_DIR / "biomed_ood_umap_grid.png"
fig.savefig(out_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved {out_path}")


## Summary

For each of the 6 `(encoder, prompt)` cells in `exp_06_biomed_text`, this notebook saves:
- `results/metrics/exp_07_ood_eval/<cell_name>_ood_classification.json`
- `results/confusion_matrices/exp_07_ood_eval/<cell_name>_ood_cm_{raw,norm}.npy`

Plus aggregate outputs:
- `results/metrics/exp_07_ood_eval/biomed_ood_summary.csv`
- `results/metrics/exp_07_ood_eval/biomed_ood_macro_f1_grid.csv`
- `results/plots/exp_07_ood_eval/biomed_ood_umap_grid.png`

**Prereq:** all 6 cells must have been trained by `exp_06_biomed_text/CLIP_Biomed_text.ipynb` first. Missing cells are skipped with a warning at evaluation time.

**Reading the grid:**
- Compare *rows* to isolate the text-encoder effect at a fixed prompt format.
- Compare *columns* to isolate the prompt effect at a fixed encoder.
- A flat grid (all cells within ~1 F1 point) → text-encoder choice doesn't move OOD performance, confirming the "frozen ImageNet ResNet50 is the bottleneck" hypothesis.
- A non-flat grid → a specific (encoder, prompt) combination provides OOD value over the baseline 0.866 — would be a positive finding worth investigating further.
